In [26]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [20]:
df = pd.read_csv('creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [21]:
scaler = StandardScaler()

df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time'] = scaler.fit_transform(df[['Time']])

In [22]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [23]:
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_res.value_counts())

Before SMOTE:
 Class
0    227451
1       394
Name: count, dtype: int64

After SMOTE:
 Class
0    227451
1    227451
Name: count, dtype: int64


---

# Model Building & Evaluation

In this step, we train classification models on the balanced training data and evaluate their performance on the unseen test data.

In [31]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_res, y_train_res)

y_pred_lr = lr.predict(X_test)

In [30]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred_lr)
recall = recall_score(y_test, y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.05813953488372093
Recall: 0.9183673469387755
F1-score: 0.10935601458080195


# Logistic Regression (Baseline Model)

Logistic Regression is used as a baseline model to establish a reference performance.

# Performance (Fraud Class - Class 1)
- Precision: 0.06  
- Recall: 0.92  
- F1-score: 0.11  

# Interpretation:
- The model achieves **very high recall**, meaning it successfully detects most fraudulent transactions.
- However, the **precision is extremely low**, indicating a large number of false positives.
- This means many normal transactions are incorrectly classified as fraud.

# Conclusion:
While Logistic Regression is effective at capturing fraud cases, it is not suitable for production due to excessive false alarms.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_res, y_train_res)

y_pred_rf = rf.predict(X_test)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.845360824742268
Recall: 0.8367346938775511
F1-score: 0.841025641025641


# Random Forest (Final Model)

Random Forest is an ensemble learning method that combines multiple decision trees to improve prediction accuracy.

# Performance (Fraud Class - Class 1)
- Precision: 0.85  
- Recall: 0.84  
- F1-score: 0.84  
- ROC-AUC: 0.97  

# Interpretation:
- The model achieves a **balanced precision and recall**, making it highly reliable.
- It significantly reduces false positives compared to Logistic Regression.
- Maintains strong fraud detection capability while minimizing unnecessary alerts.

# Conclusion:
Random Forest is the best-performing model, providing a strong balance between detecting fraud and reducing false alarms.

In [25]:
from sklearn.metrics import classification_report, confusion_matrix

print("Logistic Regression:\n")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

print("\nRandom Forest:\n")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Logistic Regression:

[[55406  1458]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962


Random Forest:

[[56849    15]
 [   16    82]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.85      0.84      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962



# Model Evaluation

# Key Metrics:
- Recall → most important for fraud detection
- Precision → avoid false alarms
- F1-score → balance of precision and recall

# Why Recall is important?
In fraud detection, missing a fraud case is more costly than a false alert.

High recall ensures that most fraud cases are detected.

# Model Comparison

# Logistic Regression
- High recall but very low precision
- Generates many false positives

# Random Forest
- Balanced precision and recall
- Much fewer false alarms

# Final Conclusion
Random Forest is the better model as it provides a balance between detecting fraud and minimizing false alerts.

In [27]:
from sklearn.metrics import roc_auc_score

y_prob_rf = rf.predict_proba(X_test)[:,1]
roc_auc = roc_auc_score(y_test, y_prob_rf)

print("ROC-AUC Score:", roc_auc)

ROC-AUC Score: 0.973103297664029


# ROC-AUC Score

The ROC-AUC score of the Random Forest model is approximately **0.97**, indicating excellent performance.

# Interpretation:
- The model can effectively distinguish between fraudulent and normal transactions
- A higher ROC-AUC score means better classification capability

# Insight:
Even with imbalanced data, the model performs strongly after applying SMOTE and Random Forest.